## 7_elevation_calculation
### This notebook downloads and processes downloaded elevation rasters from source
### Result of this notebook is a new column in DataFrame with gathered elevation for every building

### To limit the data downloading change the boundaries defined by the country polygon in cell 2

### To remove connection to IBM COS save the boundary of the area of interest on a local computer and 
 - Ignore the 4rd cell starting with "cos_client = ibm_boto3.client"
 - In the last cell remove the re-upload of intermediate results

### Initial configuration
#### To start working with this particular notebook, you need to provide necessary credential and settings
#### Below is an template of configuration, which is necessary prepare aside of this notebook and copy & paste all content in triple quotes to the next cell's input field
    """
    {
    "COUNTRY_NAME": "Kenya",
    "SEGREGATION_STYLE": "overview",

    "COS_ENDPOINT_URL": "s3.private.eu-de.cloud-object-storage.appdomain.cloud",
    "COS_AUTH_ENDPOINT_URL": "https://iam.cloud.ibm.com/oidc/token",
    "COS_APIKEY": "xxx",
    "PARQUET_BUCKET": "XXX",
    "COUNTRY_PARQUET_FILE": "FEATURES_DB_VIDA_EXTENDED.parquet",
    }
    """


In [ ]:
# Read notebook configuration
import getpass
import json

config_str = getpass.getpass('Enter your prepared config: ')
config = json.loads(config_str)

In [ ]:
# import necessary libraries
import numpy as np
import struct
import geopandas as gpd
import pandas as pd
import shapely
import os
import json
import io
from shapely.geometry import Point, Polygon, MultiPolygon
import ibm_boto3
from botocore.client import Config

import requests
import tarfile
import math
import shutil


In [2]:
# 2. Define the bounding coordinates for selected country
min_latitude = 21.877700443809616   # Minimum latitude (South)
max_latitude = 24.58632376777544  # Maximum latitude (North)\
min_longitude = 92.10795715482573  # Minimum longitude (East)
max_longitude = 93.524318295645  # Maximum longitude (West)

In [ ]:
parquet_datasets_folder = 'parquet_datasets'

cos_client = ibm_boto3.client(service_name='s3',
                              ibm_api_key_id=config["COS_APIKEY"],
                              config=Config(signature_version='oauth'),
                              endpoint_url=config["COS_ENDPOINT_URL"])

try:
    streaming_body_1 = cos_client.get_object(Bucket=config["PARQUET_BUCKET"], Key=config["COUNTRY_PARQUET_FILE"])['Body']
    print("Copying to localStorage :  " + config["COUNTRY_PARQUET_FILE"])
    with io.FileIO(config["COUNTRY_PARQUET_FILE"], 'w') as file:
        for i in io.BytesIO(streaming_body_1.read()):
            file.write(i)
    shutil.copyfile(config["COUNTRY_PARQUET_FILE"], parquet_datasets_folder)
            
except Exception as e:
    print('Error occured: ', e)

parquet_datasets = os.listdir(parquet_datasets_folder)
parquet_datasets

NameError: name 'config' is not defined

In [ ]:
# 3. Round the coordinates (min down to nearest 5, max up to nearest 5)
def round_down_to_nearest_5(value):
    return math.floor(value / 5.0) * 5

def round_up_to_nearest_5(value):
    return math.ceil(value / 5.0) * 5

# Round the min latitude/longitude down and max latitude/longitude up
rounded_min_lat = round_down_to_nearest_5(min_latitude)
rounded_max_lat = round_up_to_nearest_5(max_latitude)
rounded_min_lon = round_down_to_nearest_5(min_longitude)
rounded_max_lon = round_up_to_nearest_5(max_longitude)

# 4. Get the set of numbers in between the min and max latitudes and longitudes, inclusive
latitudes = list(range(rounded_min_lat, rounded_max_lat + 5, 5))
longitudes = list(range(rounded_min_lon, rounded_max_lon + 5, 5))

# Function to add leading zeros and "N" or "S" prefix for latitudes
def add_latitude_format(lat):
    if lat < 0:
        return f"S{abs(lat):02d}"
    else:
        return f"N{lat:02d}"

# Function to add leading zeros and "E" prefix for longitude
def add_longitude_format(lon):
    # Check if the longitude has two digits and add a leading zero, otherwise return as is
    if -100 < lon < 100:  # This ensures the number is two digits
        return f"E{lon:03d}"  # Add leading zero if it's less than 100
    else:
        return f"E{lon}"  # No leading zero for 3 digits or more

# 5. Loop over the latitudes and longitudes to generate URLs
base_url = "http://mirrors.iplantcollaborative.org/earthenv_dem_data/EarthEnv-DEM90/"

# 6. Loop over the latitudes and longitudes to download, extract, and upload the necessary files
for lat in latitudes:
    for lon in longitudes:
        # Generate the tile name (Latitude and Longitude) dynamically
        lat_formatted = add_latitude_format(lat)
        lon_formatted = add_longitude_format(lon)
        
        # Create the full URL for the file
        file_url = f"{base_url}EarthEnv-DEM90_{lat_formatted}{lon_formatted}.tar.gz"
        file_name = file_url.split("/")[-1]  # Extract the file name from the URL
        
        print(f"Downloading {file_name}...")

        # Step 1: Download the file from the URL
        response = requests.get(file_url)
        if response.status_code == 200:
            # Save the tar.gz file locally
            with open(file_name, 'wb') as f:
                f.write(response.content)
            print(f"Downloaded {file_name} successfully.")
            
            # Step 2: Extract the tar file
            extraction_path = file_name.replace('.tar.gz', '')  # Path where to extract
            with tarfile.open(file_name, 'r:gz') as tar:
                # Extract the tar archive to the specified path
                tar.extractall(path=extraction_path)
                print(f"Extracted {file_name} successfully.")
                
                # Step 3: Filter and save only the .bil, .hdr, and .prj files
                extracted_files = os.listdir(extraction_path)
                
                for extracted_file in extracted_files:
                    if extracted_file.endswith(('.bil', '.hdr', '.prj')):
                        file_path = os.path.join(extraction_path, extracted_file)
                        
                        # Specify your local directory where to save the file
                        local_file_path = os.path.join(r'elevation_rasters', extracted_file)
                        
                        # Ensure the local directory exists
                        os.makedirs(os.path.dirname(local_file_path), exist_ok=True)
                        
                        print(f"Saving {extracted_file} locally...")
                        # Save the filtered files locally
                        with open(local_file_path, 'wb') as local_file:
                            with open(file_path, 'rb') as data:
                                local_file.write(data.read())  # Write the content to the local file
                        
                        print(f"Saved {extracted_file} to {local_file_path} successfully.")


Downloaded EarthEnv-DEM90_N20E090.tar.gz successfully.
Extracted EarthEnv-DEM90_N20E090.tar.gz successfully.
Saving EarthEnv-DEM90_N20E090.bil locally...
Saved EarthEnv-DEM90_N20E090.bil to elevation_rasters\EarthEnv-DEM90_N20E090.bil successfully.
Saving EarthEnv-DEM90_N20E090.hdr locally...
Saved EarthEnv-DEM90_N20E090.hdr to elevation_rasters\EarthEnv-DEM90_N20E090.hdr successfully.
Saving EarthEnv-DEM90_N20E090.prj locally...
Saved EarthEnv-DEM90_N20E090.prj to elevation_rasters\EarthEnv-DEM90_N20E090.prj successfully.
Downloaded EarthEnv-DEM90_N20E095.tar.gz successfully.
Extracted EarthEnv-DEM90_N20E095.tar.gz successfully.
Saving EarthEnv-DEM90_N20E095.bil locally...
Saved EarthEnv-DEM90_N20E095.bil to elevation_rasters\EarthEnv-DEM90_N20E095.bil successfully.
Saving EarthEnv-DEM90_N20E095.hdr locally...
Saved EarthEnv-DEM90_N20E095.hdr to elevation_rasters\EarthEnv-DEM90_N20E095.hdr successfully.
Saving EarthEnv-DEM90_N20E095.prj locally...
Saved EarthEnv-DEM90_N20E095.prj to e

In [ ]:


def read_hdr_file(hdr_file):
    """Reads the .hdr file and extracts metadata."""
    metadata = {}
    with open(hdr_file, 'r') as file:
        for line in file:
            parts = line.strip().split()
            if len(parts) == 2:
                metadata[parts[0].upper()] = parts[1]
    return metadata

def read_bil_file(bil_file, nrows, ncols, nodata_value):
    """Reads the .bil file and returns the elevation data as a numpy array."""
    with open(bil_file, 'rb') as file:
        # Read the binary data
        raw_data = file.read()
    
    # Since the data is 16-bit signed integers, unpack the binary data accordingly
    data = struct.iter_unpack('h', raw_data)  # 'h' for 16-bit signed integer
    elevation_data = np.array([e[0] for e in data]).reshape((nrows, ncols))
    
    # Replace NODATA values with np.nan for easier handling
    nodata_value = int(nodata_value)  # Ensure nodata_value is treated as an integer
    elevation_data = np.where(elevation_data == nodata_value, np.nan, elevation_data)
    
    return elevation_data

def get_elevation_from_coordinates(coord, ulxmap, ulymap, xdim, ydim, elevation_data, ncols, nrows):
    """Converts coordinates to pixel indices and retrieves the elevation."""
    # Convert from geographic coordinates (longitude, latitude) to pixel indices
    x_pixel = int((coord[0] - ulxmap) / xdim)
    y_pixel = int((ulymap - coord[1]) / ydim)
    
    # Ensure the pixel indices are within bounds
    if 0 <= x_pixel < ncols and 0 <= y_pixel < nrows:
        return elevation_data[y_pixel, x_pixel]
    else:
        return np.nan  # Outside the raster bounds, return NaN

def update_geoparquet_with_elevation(gdf, hdr_file, bil_file):
    """Update GeoParquet records with corresponding elevation values from the .bil file."""
    # Read the .hdr file to extract metadata
    metadata = read_hdr_file(hdr_file)
    
    # Extract necessary values from metadata
    nrows = int(metadata['NROWS'])
    ncols = int(metadata['NCOLS'])
    nodata_value = metadata['NODATA']
    ulxmap = float(metadata['ULXMAP'])
    ulymap = float(metadata['ULYMAP'])
    xdim = float(metadata['XDIM'])
    ydim = float(metadata['YDIM'])
    
    # print('metadata: ', metadata)
    # Read the .bil file to extract elevation data
    elevation_data = read_bil_file(bil_file, nrows, ncols, nodata_value)
    
    # Create a new column for elevation
    missing_geometries = []  # List to track missing records for debugging

    def get_elevation_if_valid(geom):
        """Check if a valid elevation exists and return it, otherwise keep original."""
        if isinstance(geom, Point):
            # For Point geometries, directly use the coordinates
            coord = (geom.x, geom.y)
            reason = "Point"
        elif isinstance(geom, Polygon):
            # For Polygon geometries, use the centroid (the "center" point of the polygon)
            coord = (geom.centroid.x, geom.centroid.y)
            reason = "Polygon"
        elif isinstance(geom, MultiPolygon):
            # For MultiPolygon geometries, calculate the centroid of the entire MultiPolygon
            coord = (geom.centroid.x, geom.centroid.y)
            reason = "MultiPolygon"
        else:
            reason = f"Unsupported geometry: {type(geom).__name__}"  # For unsupported geometry types
            missing_geometries.append((geom, reason))
            return np.nan  # For unsupported geometry types, return NaN
        
        # Get the elevation for the given coordinate
        elevation = get_elevation_from_coordinates(
            coord, ulxmap, ulymap, xdim, ydim, elevation_data, ncols, nrows)
        
        # Check if elevation is missing (NaN)
        if np.isnan(elevation):
            if 0 <= int((coord[0] - ulxmap) / xdim) < ncols and 0 <= int((ulymap - coord[1]) / ydim) < nrows:
                reason = "NoData value or outside raster bounds"
            else:
                reason = "Out of bounds"
            missing_geometries.append((geom, reason))
            return np.nan
        else:
            return elevation

    # Apply the function to each geometry in the GeoDataFrame
    gdf['elevation'] = gdf['geometry'].apply(get_elevation_if_valid)
    
    # Print debugging information if there are missing records
    if missing_geometries:
        print(f"Missing or unsupported geometries: {len(missing_geometries)}")
        for geom, reason in missing_geometries:
            print(f"Geometry: {geom} - Reason: {reason}")
    
    return gdf



In [ ]:
elevation_files_folder = 'elevation_rasters'
all_elevation_files = os.listdir(elevation_files_folder)

In [ ]:
all_elevation_raster_names = [i.replace('.bil', '').replace('.hdr', '') for i in all_elevation_files if '.bil' in i]

elevation_rasters_files = {}

for name in all_elevation_raster_names:
    
    raster_files = {}
    bil_file = f'{name}.bil' if f'{name}.bil' in all_elevation_files else False
    
    if bil_file:
        raster_files['bil'] = bil_file
    
    hdr_file = f'{name}.hdr' if f'{name}.hdr' in all_elevation_files else False
     
    if hdr_file:
        raster_files['hdr'] = hdr_file
        
    elevation_rasters_files[name] = raster_files

In [ ]:
for k,v in elevation_rasters_files.items():
    
    if len(v) != 2:
        print(f'\033[91mMissing raster component for: {k}\033[00m')
    else: print(f'\033[92mOK: {k}\033[00m')

OK: EarthEnv-DEM90_N20E090
OK: EarthEnv-DEM90_N20E095
OK: EarthEnv-DEM90_N25E090
OK: EarthEnv-DEM90_N25E095


In [ ]:
elevation_rasters_files

{'EarthEnv-DEM90_N20E090': {'bil': 'EarthEnv-DEM90_N20E090.bil',
  'hdr': 'EarthEnv-DEM90_N20E090.hdr'},
 'EarthEnv-DEM90_N20E095': {'bil': 'EarthEnv-DEM90_N20E095.bil',
  'hdr': 'EarthEnv-DEM90_N20E095.hdr'},
 'EarthEnv-DEM90_N25E090': {'bil': 'EarthEnv-DEM90_N25E090.bil',
  'hdr': 'EarthEnv-DEM90_N25E090.hdr'},
 'EarthEnv-DEM90_N25E095': {'bil': 'EarthEnv-DEM90_N25E095.bil',
  'hdr': 'EarthEnv-DEM90_N25E095.hdr'}}

In [ ]:
for k, v in elevation_rasters_files.items():
    
    hdr_filepath = os.path.join(elevation_files_folder, v.get('hdr', ''))
    hdr_filepath
    print(hdr_filepath)

elevation_rasters\EarthEnv-DEM90_N20E090.hdr
elevation_rasters\EarthEnv-DEM90_N20E095.hdr
elevation_rasters\EarthEnv-DEM90_N25E090.hdr
elevation_rasters\EarthEnv-DEM90_N25E095.hdr


In [ ]:


for k, v in elevation_rasters_files.items():
    
    v_temp = v
    hdr_filepath = os.path.join(elevation_files_folder, v.get('hdr', ''))
    
    metadata = read_hdr_file(hdr_filepath)

    metadata['ULXMAP'] = round(float(metadata['ULXMAP']), 3)
    metadata['ULYMAP'] = round(float(metadata['ULYMAP']), 3)
    metadata['NROWS'] = float(metadata['NROWS'])
    metadata['NCOLS'] = float(metadata['NCOLS'])
    metadata['YDIM'] = float(metadata['YDIM'])
    metadata['XDIM'] = float(metadata['XDIM'])
    

    poly = Polygon(
        [
            [metadata['ULXMAP'], metadata['ULYMAP']],
            [metadata['ULXMAP'] + (metadata['XDIM'] * metadata['NROWS']), metadata['ULYMAP']],
            [metadata['ULXMAP'] + (metadata['XDIM'] * metadata['NROWS']), metadata['ULYMAP'] - (metadata['YDIM'] * metadata['NCOLS'])],
            [metadata['ULXMAP'], metadata['ULYMAP'] - (metadata['YDIM'] * metadata['NCOLS'])],
            [metadata['ULXMAP'], metadata['ULYMAP']],
        ]
    )
    
    v_temp['bbox'] = poly
    elevation_rasters_files[k] = v_temp
    

In [ ]:
def get_parquet_elevation_rasters(elevation_rasters_files, df_lat_min, df_lat_max, df_lon_min, df_lon_max):
    
    lat_range = np.arange(df_lat_min, df_lat_max, 0.2)
    lon_range = np.arange(df_lon_min, df_lon_max, 0.2)
    
    lat_range = np.append(lat_range, [df_lat_max])
    lon_range = np.append(lon_range, [df_lon_max])
    
    rasters = []
    for lat in lat_range:
        for lon in lon_range:
            
            point = Point(lon, lat)
            for raster_name, elevation_raster in elevation_rasters_files.items():
                
                if elevation_raster['bbox'].contains(point):
                   rasters.append(raster_name)
    
    rasters = list(set(rasters))
    result = [elevation_rasters_files[raster] for raster in rasters]

    return result

In [ ]:
get_buildings_from_bbox = lambda df, lon_min, lat_min, lon_max, lat_max: df[(lon_min <= df.longitude) & (df.longitude <= lon_max) & (lat_min <= df.latitude) & (df.latitude  <= lat_max)].copy()


In [ ]:
for parquet_filename in parquet_datasets:
    
    df = pd.read_parquet(os.path.join(parquet_datasets_folder, parquet_filename))
    df = gpd.GeoDataFrame(df, geometry=df['POLYGON_COORDINATES'].apply(shapely.wkt.loads))
    
    df_lat_min, df_lat_max = df.latitude.min(), df.latitude.max()
    df_lon_min, df_lon_max = df.longitude.min(), df.longitude.max()
    
    parquet_elevation_rasters = get_parquet_elevation_rasters(elevation_rasters_files, df_lat_min, df_lat_max, df_lon_min, df_lon_max)
    
    for elevation_raster in parquet_elevation_rasters:
    
        bil_file = os.path.join(elevation_files_folder, elevation_raster['bil'])
        hdr_file = os.path.join(elevation_files_folder, elevation_raster['hdr'])

        lon_min, lat_min, lon_max, lat_max = elevation_raster['bbox'].bounds
        
        bbox_df = get_buildings_from_bbox(df, lon_min, lat_min, lon_max, lat_max)
        
        # Update the GeoParquet with elevation data
        updated_gdf = update_geoparquet_with_elevation(bbox_df, hdr_file, bil_file)
        
        
        calculated_elevations_folder = 'calculated_elevations'
        filename = f"{elevation_raster['bil'].replace('.bil', '')}_{parquet_filename}"
        filepath = os.path.join(calculated_elevations_folder, filename)
        updated_gdf.to_parquet(filepath)

AttributeError: 'GeoDataFrame' object has no attribute 'latitude'

In [ ]:
try:
    res=cos_client.upload_file(Filename=filepath, Bucket=config["COUNTRY_PARQUET_FILE"],Key=filename)
except Exception as e:
    print(Exception, e)
else:
    print(f'{filename} succesfully uploaded')

count    725695.000000
mean       1185.287746
std         302.405012
min         444.000000
25%         952.000000
50%        1134.000000
75%        1377.000000
max        2006.000000
Name: elevation, dtype: float64